# 奇异值分解（Singular Value Decomposition）

对应课程：`phases/01-math-foundations/11-singular-value-decomposition`

> SVD 是线性代数中的“瑞士军刀”。每一个矩阵都拥有 SVD，每一位数据科学家都需要掌握它。

本 notebook 把 `svd.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `svd.py`。

**贯穿全课的模式：** 任意矩阵 $A = U\Sigma V^T$：先旋转，再按奇异值拉伸，再旋转。截断小奇异值就是压缩。


## 0. 依赖

本课源码已经用 numpy（课程允许清单内）。


In [1]:
import numpy as np

np.random.seed(42)


## 1. 幂迭代：找出最大特征对

对对称矩阵 $M$，反复 $v \leftarrow Mv/\|Mv\|$，会收敛到最大特征值对应的特征向量。SVD 对 $A^TA$ 做这件事，因为 $A^TA$ 的特征值是奇异值的平方。


In [2]:
def power_iteration(M, num_iters=200, tol=1e-10):
    """对称矩阵的最大特征值 / 特征向量。"""
    n = M.shape[1]
    v = np.random.randn(n)
    v = v / np.linalg.norm(v)
    for _ in range(num_iters):
        Mv = M @ v
        norm = np.linalg.norm(Mv)
        if norm < tol:
            return 0.0, v
        v_new = Mv / norm
        if np.abs(np.dot(v_new, v)) > 1 - tol:
            v = v_new
            break
        v = v_new
    eigenvalue = v @ M @ v
    return eigenvalue, v


M = np.array([[4.0, 1.0], [1.0, 3.0]])
lam, v = power_iteration(M)
print("特征值:", round(lam, 6), "向量:", np.round(v, 4))
print("Mv 是否平行于 v:", np.allclose(M @ v, lam * v))


特征值: 4.618034 向量: [0.8507 0.5257]
Mv 是否平行于 v: False


## 2. 从零 SVD：一次剥一层

$A^TA$ 的主特征向量是 $v_1$，对应 $\sigma_1=\sqrt{\lambda}$，$u_1 = Av_1/\sigma_1$。从 $A$ 里减掉 $\sigma_1 u_1 v_1^T$（秩一更新），再对残差重复，就是完整 SVD。


In [3]:
def svd_from_scratch(A, k=None):
    """幂迭代 + 秩一剥层。返回 U, S, V（V 的列是右奇异向量）。"""
    m, n = A.shape
    if k is None:
        k = min(m, n)
    sigmas, us, vs = [], [], []
    A_residual = A.copy().astype(float)
    for _ in range(k):
        AtA = A_residual.T @ A_residual
        eigenvalue, v = power_iteration(AtA, num_iters=300)
        if eigenvalue < 1e-10:
            break
        sigma = np.sqrt(max(eigenvalue, 0))
        u = A_residual @ v / sigma
        u_norm = np.linalg.norm(u)
        if u_norm > 1e-10:
            u = u / u_norm
        sigmas.append(sigma)
        us.append(u)
        vs.append(v)
        A_residual = A_residual - sigma * np.outer(u, v)
    U = np.column_stack(us) if us else np.empty((m, 0))
    S = np.array(sigmas)
    V = np.column_stack(vs) if vs else np.empty((n, 0))
    return U, S, V


A = np.random.randn(6, 4)
U, S, V = svd_from_scratch(A)
A_hat = U @ np.diag(S) @ V.T
print("我们的奇异值:", np.round(S, 4))
print("NumPy 奇异值:", np.round(np.linalg.svd(A, compute_uv=False), 4))
print("重构误差:", np.linalg.norm(A - A_hat))


我们的奇异值: [3.5296 2.1706 1.8015 1.3999]
NumPy 奇异值: [3.5296 2.1706 1.8015 1.3999]
重构误差: 4.888843483876897e-16


## 3. 几何：旋转 → 拉伸 → 旋转

对单位圆上的点先乘 $V^T$（对齐主轴），再按 $\sigma$ 拉伸成椭圆，最后乘 $U$ 转到输出空间。


In [4]:
A = np.array([[3.0, 1.0], [1.0, 3.0]])
U, S, Vt = np.linalg.svd(A)
print("U:\n", np.round(U, 4))
print("Sigma:", np.round(S, 4))
print("V^T:\n", np.round(Vt, 4))
print("U 正交?", np.allclose(U.T @ U, np.eye(2)))

p = np.array([1.0, 0.0])
step1 = Vt @ p
step2 = S * step1
step3 = U @ step2
print("点 [1,0] 三步:", np.round(step1, 4), np.round(step2, 4), np.round(step3, 4))
print("直接 A @ p:     ", np.round(A @ p, 4))


U:
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]
Sigma: [4. 2.]
V^T:
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]
U 正交? True
点 [1,0] 三步: [-0.7071 -0.7071] [-2.8284 -1.4142] [3. 1.]
直接 A @ p:      [3. 1.]


## 4. 截断 SVD 与压缩比

Eckart–Young：保留最大的 $k$ 个奇异值，得到在 Frobenius 范数下最优的秩-$k$ 近似。

存储从 $mn$ 变成 $k(m+n+1)$。


In [5]:
def truncated_svd(A, k):
    """NumPy SVD 后只留前 k 个成分。"""
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    return U[:, :k], S[:k], Vt[:k, :]


def reconstruct(U, S, Vt):
    """U @ diag(S) @ Vt。"""
    return U @ np.diag(S) @ Vt


def compression_ratio(m, n, k):
    """压缩后存储 / 原始存储。小于 1 才真的省空间。"""
    return (k * (m + n + 1)) / (m * n)


m, n, true_rank = 80, 60, 4
U_true = np.linalg.qr(np.random.randn(m, true_rank))[0]
V_true = np.linalg.qr(np.random.randn(n, true_rank))[0]
A = U_true @ np.diag([40.0, 20.0, 10.0, 4.0]) @ V_true.T
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("前 8 个奇异值:", np.round(S[:8], 4))
print(f"{'k':>3}  {'相对误差':>10}  {'压缩比':>8}")
A_norm = np.linalg.norm(A, "fro")
for k in range(1, 7):
    Uk, Sk, Vtk = truncated_svd(A, k)
    err = np.linalg.norm(A - reconstruct(Uk, Sk, Vtk), "fro") / A_norm
    print(f"{k:3d}  {err:10.6f}  {compression_ratio(m, n, k):7.1%}")


前 8 个奇异值: [40. 20. 10.  4.  0.  0.  0.  0.]
  k        相对误差       压缩比
  1    0.493818     2.9%
  2    0.234138     5.9%
  3    0.086957     8.8%
  4    0.000000    11.8%
  5    0.000000    14.7%
  6    0.000000    17.6%


## 5. 伪逆：最小二乘的 SVD 解

$$
A^+ = V \Sigma^{+} U^T
$$

$\sigma$ 太小就当成 0（不取倒数），避免噪声被放大。超定系统 $Ax \approx b$ 的最小二乘解是 $x = A^+ b$。


In [6]:
def pseudoinverse_via_svd(A, tol=1e-10):
    """Moore-Penrose 伪逆：1/sigma，过小的奇异值丢掉。"""
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    S_inv = np.array([1.0 / s if s > tol else 0.0 for s in S])
    return Vt.T @ np.diag(S_inv) @ U.T


A = np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
b = np.array([1.0, 2.0, 2.5])
x = pseudoinverse_via_svd(A) @ b
print("x =", np.round(x, 4))
print("残差 ||Ax-b|| =", np.linalg.norm(A @ x - b))
print("与 lstsq 一致?", np.allclose(x, np.linalg.lstsq(A, b, rcond=None)[0]))


x = [0.3333 0.75  ]
残差 ||Ax-b|| = 0.20412414523193131
与 lstsq 一致? True


## 对照表

| 函数 | 角色 |
|------|------|
| `power_iteration` | $A^TA$ 的主特征对 |
| `svd_from_scratch` | 剥层得到 $U,\Sigma,V$ |
| `truncated_svd` / `reconstruct` | 秩 $k$ 近似 |
| `compression_ratio` | $k(m+n+1)/mn$ |
| `pseudoinverse_via_svd` | 最小二乘 $A^+b$ |

图像压缩、推荐系统、LSA 的长 demo 仍在：

```bash
python svd.py
```
